<a href="https://colab.research.google.com/github/Sourav1429/Machine_Unlearning/blob/main/CNN_unlearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The working of Machine Unlearning by corrupting data and then making it forget

1) DTC

2) SVM

3) CNN

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
import cv2
import os
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense
import tensorflow as tf

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
class type_of_data:
  def __init__(self,choice,path):
    self.c = choice
    self.p = path
  def return_data(self):
    X,y = [],[]
    if(self.c==1):
      data = pd.read_excel(self.p)
      X,y = data[data.columns[0:-1]].values,data[data.columns[-1]].values
    elif(self.c==2):
      labels = os.listdir(path)
      for i in range(len(labels)):
        complete_path = os.path.join(self.p,labels[i])
        for j in os.listdir(complete_path):
          img = cv2.imread(os.path.join(complete_path,j))
          img = cv2.resize(img,(128,128))
          X.append(img)
          y.append(i)
      X,y = np.array(X),np.array(y)
    elif(self.c==3):
      data = pd.read_csv(self.p)
      y = data['label'].values
      X = np.reshape(data.drop('label',axis=1).values,(len(y),28,28,1))
    return X,y

In [51]:
class define_model:
  def __init__(self,model,X_train,y_train):
    self.m = model
    self.X = X_train
    self.y = y_train
  def train(self):
    if(self.m==0):
      model = DecisionTreeClassifier()
      model.fit(self.X,self.y)
    elif(self.m==1):
      model = SVC()
      model.fit(self.X,self.y)
    elif(self.m==2):
      self.X = self.X/255
      inp_shape = self.X.shape[1:]
      out_shape = len(np.unique(self.y))
      self.y = tf.keras.utils.to_categorical(self.y,num_classes=out_shape)
      model = Sequential()
      model.add(Conv2D(32,(3,3),activation='relu',input_shape=inp_shape))
      model.add(MaxPooling2D((2,2)))
      model.add(Conv2D(64,(3,3),activation='relu'))
      model.add(MaxPooling2D((2,2)))
      model.add(Conv2D(64,(3,3),activation='relu'))
      model.add(Flatten())
      model.add(Dense(64,activation='relu'))
      model.add(Dense(out_shape,activation='softmax'))
      model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
      model.fit(self.X,self.y,epochs=5)
    return model

In [ ]:
path = "/content/drive/MyDrive/iris.xlsx"
choice = 1
data = type_of_data(choice,path)
X,y = data.return_data()
unique_labels = np.unique(y)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
model_choice = 1 # 0,1,2----- 0 - DTC, 1 - SVM , 2 - CNN
model_tr = define_model(model_choice,X_train,y_train)
model = model_tr.train()

After we have defined our models, we are going to test the models

1) Original accuracy

In [ ]:
if(model_choice==0):
  print(model.score(X_test,y_test))
elif(model_choice==1):
  print(model.score(X_test,y_test))
elif(model_choice==2):
  print(model.evaluate(X_test,y_test))
#model.predict(X_test)
#check prdiction of some data

1.0


2) Corrupt few labels and thn get the accuracy

In [ ]:
y_train_corr = y_train.copy()
ratio = 0.2 #ratio of total data points to be corrupted
size = int(ratio*len(X_train))
#print(y_train)
#print("===============")
corrupted_indices = np.array(random.sample(range(0, len(y_train)), size))
for i in corrupted_indices:
  y_train_corr[i] = np.random.choice(unique_labels)
  print(y_train[i],"======>",y_train_corr[i])
corr_model_tr = define_model(model_choice,X_train,y_train_corr)
corr_model = corr_model_tr.train()
if(model_choice==0):
  print(corr_model.score(X_test,y_test))
elif(model_choice==1):
  print(corr_model.score(X_test,y_test))
elif(model_choice==2):
  print(corr_model.evaluate(X_test,y_test))

2 ======> 0
1 ======> 1
2 ======> 0
2 ======> 1
2 ======> 2
2 ======> 1
1 ======> 0
0 ======> 2
2 ======> 0
1 ======> 1
2 ======> 1
2 ======> 1
2 ======> 2
2 ======> 1
0 ======> 0
1 ======> 1
0 ======> 2
2 ======> 1
0 ======> 0
1 ======> 2
2 ======> 0
0.9333333333333333


3) Perform Retraining by removing the corrupted data points and perform re-training

In [ ]:
#First step is deleting the not required indices
X_train_del = np.delete(X_train,corrupted_indices,axis=0)
y_train_del = np.delete(y_train,corrupted_indices,axis=0)
del_model_tr = define_model(model_choice,X_train_del,y_train_del)
del_model = del_model_tr.train()
if(model_choice==0):
  print(del_model.score(X_test,y_test))
elif(model_choice==1):
  print(del_model.score(X_test,y_test))
elif(model_choice==2):
  print(del_model.evaluate(X_test,y_test))

0.9777777777777777


4) Perform unlearning and then obtain accuracy

In [ ]:
#Creating an augmented dataset
X_train_aug,y_train_aug = [],[]
for i in range(len(corrupted_indices)):
  for j in unique_labels:
    if(j!=y_train[corrupted_indices[i]]):
      X_train_aug.append(X_train[corrupted_indices[i]])
      y_train_aug.append(j)
X_train_aug,y_train_aug = np.array(X_train_aug),np.array(y_train_aug)
X_concat = np.concatenate((X_train,X_train_aug),axis=0)
y_concat = np.concatenate((y_train,y_train_aug),axis=0)
print(X_concat.shape)
print(y_concat.shape)

(147, 4)
(147,)


In [ ]:
aug_model_tr = define_model(model_choice,X_concat,y_concat)
aug_model = aug_model_tr.train()
if(model_choice==0):
  print(aug_model.score(X_test,y_test))
elif(model_choice==1):
  print(aug_model.score(X_test,y_test))
elif(model_choice==2):
  print(aug_model.evaluate(X_test,y_test))

0.9777777777777777


2) DTC

In [ ]:
path = "/content/drive/MyDrive/iris.xlsx"
choice = 1
data = type_of_data(choice,path)
X,y = data.return_data()
unique_labels = np.unique(y)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=12)
model_choice = 0 # 0,1,2----- 0 - DTC, 1 - SVM , 2 - CNN
model_tr = define_model(model_choice,X_train,y_train)
model = model_tr.train()
print("Original model accuracy:")
if(model_choice==0):
  print(model.score(X_test,y_test))
elif(model_choice==1):
  print(model.score(X_test,y_test))
elif(model_choice==2):
  print(model.evaluate(X_test,y_test))

print("Corrupted model accuracy:")
y_train_corr = y_train.copy()
ratio = 0.2 #ratio of total data points to be corrupted
size = int(ratio*len(X_train))
#print(y_train)
#print("===============")
corrupted_indices = np.array(random.sample(range(0, len(y_train)), size))
for i in corrupted_indices:
  y_train_corr[i] = np.random.choice(unique_labels)
  print(y_train[i],"======>",y_train_corr[i])
corr_model_tr = define_model(model_choice,X_train,y_train_corr)
corr_model = corr_model_tr.train()
if(model_choice==0):
  print(corr_model.score(X_test,y_test))
elif(model_choice==1):
  print(corr_model.score(X_test,y_test))
elif(model_choice==2):
  print(corr_model.evaluate(X_test,y_test))

print("Retraining accuracy")
del_model_tr = define_model(model_choice,X_train_del,y_train_del)
del_model = del_model_tr.train()
if(model_choice==0):
  print(del_model.score(X_test,y_test))
elif(model_choice==1):
  print(del_model.score(X_test,y_test))
elif(model_choice==2):
  print(del_model.evaluate(X_test,y_test))


print("Unlearning accuracy")
aug_model_tr = define_model(model_choice,X_concat,y_concat)
aug_model = aug_model_tr.train()
if(model_choice==0):
  print(aug_model.score(X_test,y_test))
elif(model_choice==1):
  print(aug_model.score(X_test,y_test))
elif(model_choice==2):
  print(aug_model.evaluate(X_test,y_test))

Original model accuracy:
0.9333333333333333
Corrupted model accuracy:
2 ======> 2
1 ======> 0
1 ======> 1
1 ======> 0
2 ======> 1
2 ======> 0
2 ======> 1
2 ======> 0
1 ======> 1
2 ======> 1
1 ======> 2
2 ======> 2
0 ======> 2
1 ======> 2
2 ======> 1
1 ======> 0
0 ======> 2
0 ======> 1
1 ======> 0
1 ======> 1
1 ======> 1
1 ======> 2
1 ======> 1
2 ======> 2
0.6333333333333333
Retraining accuracy
1.0
Unlearning accuracy
0.8333333333333334


In [10]:
###Using image classification data
path = "/content/drive/MyDrive/mnist_train.csv"
choice = 3
data = type_of_data(choice,path)
X,y = data.return_data()
unique_labels = np.unique(y)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
model_choice = 2 # 0,1,2----- 0 - DTC, 1 - SVM , 2 - CNN
model_tr = define_model(model_choice,X_train,y_train)
model = model_tr.train()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 48s 35ms/step - accuracy: 0.8691 - loss: 0.4245
Epoch 2/10
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 46s 35ms/step - accuracy: 0.9829 - loss: 0.0550
Epoch 3/10
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 81s 34ms/step - accuracy: 0.9882 - loss: 0.0358
Epoch 4/10
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 83s 35ms/step - accuracy: 0.9917 - loss: 0.0265
Epoch 5/10
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 84s 36ms/step - accuracy: 0.9936 - loss: 0.0195
Epoch 6/10
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 79s 34ms/step - accuracy: 0.9943 - loss: 0.0174
Epoch 7/10
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 83s 35ms/step - accuracy: 0.9953 - loss: 0.0140
Epoch 8/10
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 46s 35ms/step - accuracy: 0.9967 - loss: 0.0108
Epoch 9/10
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 45s 34ms/step - accuracy: 0.9967 - loss: 0.0091
Epoch 10/10
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 44s 33ms/step - accuracy: 0.9959 - loss: 0.0105


In [14]:
#y_test = tf.keras.utils.to_categorical(y_test,num_classes=unique_labels)
if(model_choice==0):
  print(model.score(X_test,y_test))
elif(model_choice==1):
  print(model.score(X_test,y_test))
elif(model_choice==2):
  y_test = tf.keras.utils.to_categorical(y_test,num_classes=10)
  print(model.evaluate(X_test,y_test))

563/563 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9863 - loss: 16.8223
[15.247727394104004, 0.9863333106040955]


Accuracy: 98.6% and cumulative loss is 15.24

In [52]:
#Let us now corrupt a few labels and then check
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
y_train_corr = y_train.copy()
ratio = 0.02 #ratio of total data points to be corrupted
size = int(ratio*len(X_train))
#print(y_train)
#print("===============")
corrupted_indices = np.array(random.sample(range(0, len(y_train)), size))
for i in corrupted_indices:
  y_train_corr[i] = (y_train[i]+5)%len(unique_labels)
  #print(y_train[i],"======>",y_train_corr[i])
corr_model_tr = define_model(model_choice,X_train,y_train_corr)
corr_model = corr_model_tr.train()
#X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.02,random_state=12)
if(model_choice==0):
  print(corr_model.score(X_test,y_test))
elif(model_choice==1):
  print(corr_model.score(X_test,y_test))
elif(model_choice==2):
  y_test = tf.keras.utils.to_categorical(y_test,num_classes=10)
  print(corr_model.evaluate(X_test,y_test))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 49s 36ms/step - accuracy: 0.8490 - loss: 0.5383
Epoch 2/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 46s 35ms/step - accuracy: 0.9609 - loss: 0.1693
Epoch 3/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 81s 34ms/step - accuracy: 0.9712 - loss: 0.1359
Epoch 4/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 82s 34ms/step - accuracy: 0.9715 - loss: 0.1300
Epoch 5/5
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 46s 35ms/step - accuracy: 0.9741 - loss: 0.1186
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9853 - loss: 2.6945
[4.232728004455566, 0.9800000190734863]


In [54]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
if(model_choice==0):
  print(corr_model.score(X_test,y_test))
elif(model_choice==1):
  print(corr_model.score(X_test,y_test))
elif(model_choice==2):
  y_test = tf.keras.utils.to_categorical(y_test,num_classes=10)
  print(corr_model.evaluate(X_test,y_test))

563/563 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9563 - loss: 26.5855
[26.446935653686523, 0.9566110968589783]


Loss =146.78 and accuracy = 76.7%

In [35]:
#We now delete the corrupted datapoints and retrain
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=12)
#y_train_corr = y_train.copy()
ratio = 0.2 #ratio of total data points to be corrupted
size = int(ratio*len(X_train))
#print(y_train)
#print("===============")
corrupted_indices = np.array(random.sample(range(0, len(y_train)), size))
#print(y_train[i],"======>",y_train_corr[i])
X_train_del = np.delete(X_train,corrupted_indices,axis=0)
y_train_del = np.delete(y_train,corrupted_indices,axis=0)
del_model_tr = define_model(model_choice,X_train_del,y_train_del)
del_model = del_model_tr.train()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 45s 35ms/step - accuracy: 0.8549 - loss: 0.4555
Epoch 2/10
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 40s 34ms/step - accuracy: 0.9822 - loss: 0.0551
Epoch 3/10
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 42s 35ms/step - accuracy: 0.9862 - loss: 0.0433
Epoch 4/10
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 83s 35ms/step - accuracy: 0.9907 - loss: 0.0295
Epoch 5/10
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 79s 33ms/step - accuracy: 0.9934 - loss: 0.0217
Epoch 6/10
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 44s 36ms/step - accuracy: 0.9939 - loss: 0.0201
Epoch 7/10
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 79s 34ms/step - accuracy: 0.9955 - loss: 0.0133
Epoch 8/10
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 41s 34ms/step - accuracy: 0.9959 - loss: 0.0118
Epoch 9/10
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 41s 34ms/step - accuracy: 0.9949 - loss: 0.0129
Epoch 10/10
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 40s 33ms/step - accuracy: 0.9975 - loss: 0.0085


ValueError: Arguments `target` and `output` must have the same rank (ndim). Received: target.shape=(32,), output.shape=(32, 10)

In [36]:
if(model_choice==0):
  print(del_model.score(X_test,y_test))
elif(model_choice==1):
  print(del_model.score(X_test,y_test))
elif(model_choice==2):
  y_test = tf.keras.utils.to_categorical(y_test,num_classes=10)
  print(del_model.evaluate(X_test,y_test))

375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.9865 - loss: 11.1080
[11.179784774780273, 0.9872499704360962]


In [42]:
#Perform unlearning
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=12)
ratio = 0.002 #ratio of total data points to be corrupted
size = int(ratio*len(X_train))
#print(y_train)
#print("===============")
corrupted_indices = np.array(random.sample(range(0, len(y_train)), size))
X_train_aug,y_train_aug = [],[]
for i in range(len(corrupted_indices)):
  for j in unique_labels:
    if(j!=y_train[corrupted_indices[i]]):
      X_train_aug.append(X_train[corrupted_indices[i]])
      y_train_aug.append(j)
X_train_aug,y_train_aug = np.array(X_train_aug),np.array(y_train_aug)
X_concat = np.concatenate((X_train,X_train_aug),axis=0)
y_concat = np.concatenate((y_train,y_train_aug),axis=0)
print(X_concat.shape)
print(y_concat.shape)

(48864, 28, 28, 1)
(48864,)


In [43]:
aug_model_tr = define_model(model_choice,X_concat,y_concat)
aug_model = aug_model_tr.train()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1527/1527 ━━━━━━━━━━━━━━━━━━━━ 54s 34ms/step - accuracy: 0.8677 - loss: 0.5028
Epoch 2/10
1527/1527 ━━━━━━━━━━━━━━━━━━━━ 82s 34ms/step - accuracy: 0.9658 - loss: 0.1794
Epoch 3/10
1527/1527 ━━━━━━━━━━━━━━━━━━━━ 84s 35ms/step - accuracy: 0.9712 - loss: 0.1335
Epoch 4/10
1527/1527 ━━━━━━━━━━━━━━━━━━━━ 82s 36ms/step - accuracy: 0.9747 - loss: 0.1010
Epoch 5/10
1527/1527 ━━━━━━━━━━━━━━━━━━━━ 58s 38ms/step - accuracy: 0.9749 - loss: 0.0932
Epoch 6/10
1527/1527 ━━━━━━━━━━━━━━━━━━━━ 54s 36ms/step - accuracy: 0.9792 - loss: 0.0741
Epoch 7/10
1527/1527 ━━━━━━━━━━━━━━━━━━━━ 53s 35ms/step - accuracy: 0.9790 - loss: 0.0666
Epoch 8/10
1527/1527 ━━━━━━━━━━━━━━━━━━━━ 83s 35ms/step - accuracy: 0.9793 - loss: 0.0613
Epoch 9/10
1527/1527 ━━━━━━━━━━━━━━━━━━━━ 83s 36ms/step - accuracy: 0.9800 - loss: 0.0583
Epoch 10/10
1527/1527 ━━━━━━━━━━━━━━━━━━━━ 79s 35ms/step - accuracy: 0.9787 - loss: 0.0614


In [44]:
if(model_choice==0):
  print(aug_model.score(X_test,y_test))
elif(model_choice==1):
  print(aug_model.score(X_test,y_test))
elif(model_choice==2):
  y_test = tf.keras.utils.to_categorical(y_test,num_classes=10)
  print(aug_model.evaluate(X_test,y_test))

375/375 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9894 - loss: 8.3163
[8.244504928588867, 0.9891666769981384]
